In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy

2025-12-03 08:44:44.503584: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-03 08:44:44.503637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-03 08:44:44.526965: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-03 08:44:44.575747: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 08:44:45.662950: W tensorflow/comp

In [2]:
batch_size = 128
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [ ]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.c = [tf.zeros_like(v) for v in self.model.variables]
    def upload(self, delta_ws, delta_cs):
        for v, dw in zip(self.model.variables, delta_ws):
            v.assign_add(dw)
        for c_global, dc in zip(self.c, delta_cs):
            c_global.assign_add(dc)
        return self.model, self.c
    def download(self):
        return self.model, self.c
    def initModel(self, x):
        self.model(x)

In [5]:
def valiAll():
    m = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv.append(va_r2.numpy())

In [ ]:
class Node:
    def __init__(self, dsName,freq):
        self.model = MLP()
        self.freq = freq
        dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
        self.c_i = None
    def train(self, num_epochs):
        for round_idx in range(num_epochs):
            # 1. 从服务器下载当前全局模型参数 w_t 和全局控制变量 c
            global_model, c_global = ps.download()
            self.model = copy.deepcopy(global_model)

            # 初始化本地 c_i
            if self.c_i is None:
                self.c_i = [tf.zeros_like(v) for v in self.model.variables]

            # 2. 保存本轮开始时的参数 w_t
            w_before = [tf.identity(v) for v in self.model.variables]

            step = 0
            last_tr_mse = None
            last_tr_rmse = None
            last_tr_mae = None
            last_tr_r2 = None

            # 3. 本地用修正梯度做 K 步更新
            for X, y in self.dataset_train:

                with tf.GradientTape() as tape:
                    y_pred = self.model(X)
                    tr_mse = tf.reduce_mean(tf.square(y_pred - y))

                grads = tape.gradient(tr_mse, self.model.variables)

                # 修正梯度：g - c_i + c
                corrected_grads = [
                    g - ci + cg
                    for g, ci, cg in zip(grads, self.c_i, c_global)
                ]

                # 用简单 SGD 更新本地模型参数
                for v, g_corr in zip(self.model.variables, corrected_grads):
                    v.assign_sub(learning_rate * g_corr)

                # 记录最后一个 batch 的指标（方便打印）
                last_tr_mse = tr_mse
                last_tr_rmse = tf.sqrt(tr_mse)
                last_tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
                last_tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(
                    tf.square(y - tf.reduce_mean(y))
                )

            # 4. 本地更新完成后，计算 Δw_i
            w_after = [tf.identity(v) for v in self.model.variables]
            delta_w = [
                w_a - w_b
                for w_a, w_b in zip(w_after, w_before)
            ]

            # 5. 按 SCAFFOLD 公式更新本地控制变量 c_i，并得到 Δc_i
            #    c_i_new = c_i - c + (1 / (K * η)) * (w_before - w_after)
            K = max(step, 1)  # 防止除零
            scale = 1.0 / (K * learning_rate)

            new_c_i = []
            delta_c_i = []
            for ci_old, c_g, w_b, w_a in zip(self.c_i, c_global, w_before, w_after):
                ci_new = ci_old - c_g + scale * (w_b - w_a)
                new_c_i.append(ci_new)
                delta_c_i.append(ci_new - ci_old)

            # 更新本地 c_i
            self.c_i = new_c_i

            # 6. 把 Δw_i 和 Δc_i 上传给服务器
            global_model, c_global = ps.upload(delta_w, delta_c_i)

            # 7. 打印训练信息 + 记录 r2 + 调用你的验证函数
            if last_tr_mse is not None:
                print("node:{} round:{}".format(self.freq, round_idx))
                print("train mse:{} rmse:{} mae:{} r2:{}".format(
                    last_tr_mse, last_tr_rmse, last_tr_mae, last_tr_r2
                ))
                r2s.append(last_tr_r2.numpy())

            # 用当前全局模型做验证（假设 valiAll 内部会用 ps.model 或保存好的全局模型）
            valiAll()

In [7]:
r2s = []
r2sv = []

In [8]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-03 08:45:00.239101: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9610 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5
2025-12-03 08:45:00.240233: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 9554 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:65:00.0, compute capability: 7.5


In [10]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [ ]:
nodeList[0].train(150)
nodeList[1].train(150)
nodeList[2].train(150)
nodeList[0].train(150)
nodeList[1].train(150)
nodeList[2].train(150)

2025-12-03 08:45:06.716307: I external/local_xla/xla/service/service.cc:168] XLA service 0x5e6510187b00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-03 08:45:06.716328: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-12-03 08:45:06.716337: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (1): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-12-03 08:45:06.763346: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-03 08:45:06.915727: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
I0000 00:00:1764751507.114816    7343 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


node:2.4 epoch:0
train mse:0.08413834869861603 rmse:0.29006609320640564 mae:0.2331370860338211 r2:0.3079913854598999
mse:0.09641411900520325 rmse:0.3105062246322632 mae:0.24880366027355194 r2:0.20189523696899414
node:2.4 epoch:1
train mse:0.08222702890634537 rmse:0.2867525517940521 mae:0.23353254795074463 r2:0.32289689779281616
mse:0.09227575361728668 rmse:0.30376923084259033 mae:0.24484451115131378 r2:0.2361520528793335
node:2.4 epoch:2
train mse:0.07342637330293655 rmse:0.27097299695014954 mae:0.21466021239757538 r2:0.3875300884246826
mse:0.08845885097980499 rmse:0.29742032289505005 mae:0.23799291253089905 r2:0.26774799823760986
node:2.4 epoch:3
train mse:0.07564359158277512 rmse:0.2750338017940521 mae:0.22266104817390442 r2:0.37503713369369507
mse:0.07810226082801819 rmse:0.2794678211212158 mae:0.22703884541988373 r2:0.3534786105155945
node:2.4 epoch:4
train mse:0.0635000616312027 rmse:0.2519921660423279 mae:0.20505034923553467 r2:0.47451215982437134
mse:0.07216297090053558 rmse:0.2

In [21]:
for i in r2sv:
    print(i)

0.05343145
0.21146125
0.2544238
0.2703812
0.2867133
0.30649197
0.31547016
0.3224408
0.3291536
0.3452314
0.34077
0.3562141
0.354967
0.35245204
0.32595384
0.40231603
0.40158945
0.40962702
0.42272305
0.4230671
0.4107265
0.41005546
0.436269
0.44843858
0.43839198
0.43388087
0.45969415
0.45770818
0.4541483
0.4664585
0.47532052
0.46199465
0.47981328
0.47990716
0.49459165
0.4690683
0.45069534
0.48096782
0.49847054
0.46773773
0.4992084
0.47926748
0.49972522
0.48676974
0.4963382
0.48902565
0.5175091
0.50038993
0.46291137
0.5333949
0.53417766
0.48639023
0.55056655
0.5456645
0.53572404
0.5499059
0.5739113
0.5726086
0.47727573
0.5278763
0.49325925
0.58120906
0.47513604
0.5654805
0.5336244
0.4959963
0.5241405
0.5895729
0.45611674
0.37396538
0.50237
0.46507394
0.513464
0.49617326
0.5751781
0.5460856
0.4678169
0.5107386
0.5489664
0.61125076
0.45940083
0.43182415
0.49852914
0.52809775
0.47194016
0.5567453
0.496157
0.3778242
0.471923
0.41169584
0.5299705
0.43949437
0.41430855
0.38863832
0.47739798
0.538